# DTS406TC — Natural Language Processing
## Lab 2: POS Tagging and Named Entity Recognition (NER)

This notebook covers:
1. POS Tagging with NLTK
2. Basic NER with NLTK's `ne_chunk`
3. Training a custom MaxEnt NER classifier
4. **Exercise 1** — Applying POS/NER to a Reddit-style dataset with extended features
5. **Exercise 2** — Domain-specific NER (Medical) using spaCy

---

## 0. Setup — Install & Download Resources

Before anything else we install `spaCy` (needed for Exercise 2) and download the required NLTK data packages.

- **`punkt`** — tokenizer models  
- **`averaged_perceptron_tagger_eng`** — the POS tagger  
- **`maxent_ne_chunker_tab`** — the built-in NER chunker  
- **`words`** — English word list used internally by the chunker

In [1]:
# Install spaCy and its small English model (needed for Exercise 2)
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')

print("All NLTK resources downloaded.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\A.Mamah25\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\A.Mamah25\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


All NLTK resources downloaded.


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\A.Mamah25\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     C:\Users\A.Mamah25\AppData\Roaming\nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\A.Mamah25\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!


---
## Part 1 — POS Tagging with NLTK

**Part-of-Speech (POS) tagging** assigns a grammatical label to every token in a sentence.

| Tag | Meaning | Example |
|-----|---------|--------|
| DT  | Determiner | the, a |
| NN  | Noun (singular) | cat, dog |
| NNS | Noun (plural) | cats, dogs |
| VB  | Verb (base form) | run |
| VBZ | Verb (3rd person singular present) | runs |
| VBD | Verb (past tense) | ran |
| JJ  | Adjective | big, red |
| RB  | Adverb | quickly |
| IN  | Preposition | in, on, after |
| PRP | Pronoun | he, she, it |

The pipeline is: **raw text → tokenise → POS tag**.

In [3]:
from nltk import pos_tag
from nltk.tokenize import word_tokenize

# Example sentence from the slides
text = "The hungry grey cat runs after the little mouse."

# Step 1: tokenise into individual words
tokens = word_tokenize(text)
print("Tokens:", tokens)

# Step 2: assign a POS tag to each token
tagged = pos_tag(tokens)
print("\nTagged:", tagged)

Tokens: ['The', 'hungry', 'grey', 'cat', 'runs', 'after', 'the', 'little', 'mouse', '.']

Tagged: [('The', 'DT'), ('hungry', 'JJ'), ('grey', 'NN'), ('cat', 'NN'), ('runs', 'VBZ'), ('after', 'IN'), ('the', 'DT'), ('little', 'JJ'), ('mouse', 'NN'), ('.', '.')]


### 1.1 Pretty-printing the tags

The raw output is a list of `(word, tag)` tuples. Let's format it into a readable table.

In [4]:
print(f"{'Word':<12} {'POS Tag':<8} {'Meaning'}")
print("-" * 40)

# Map common tags to human-readable descriptions
tag_meanings = {
    'DT': 'Determiner', 'NN': 'Noun (singular)', 'NNS': 'Noun (plural)',
    'VB': 'Verb (base)', 'VBZ': 'Verb (3rd sg. pres.)', 'VBD': 'Verb (past tense)',
    'VBN': 'Verb (past participle)', 'JJ': 'Adjective', 'RB': 'Adverb',
    'IN': 'Preposition', 'PRP': 'Pronoun', '.': 'Punctuation'
}

for word, tag in tagged:
    meaning = tag_meanings.get(tag, tag)
    print(f"{word:<12} {tag:<8} {meaning}")

Word         POS Tag  Meaning
----------------------------------------
The          DT       Determiner
hungry       JJ       Adjective
grey         NN       Noun (singular)
cat          NN       Noun (singular)
runs         VBZ      Verb (3rd sg. pres.)
after        IN       Preposition
the          DT       Determiner
little       JJ       Adjective
mouse        NN       Noun (singular)
.            .        Punctuation


### 1.2 POS Tagging on multiple sentences

Let's test disambiguation — the word **"record"** has different POS tags depending on usage.

In [5]:
sentences = [
    "Please record the meeting for later review.",   # record = verb
    "She broke the world record last year."           # record = noun
]

for sent in sentences:
    tokens = word_tokenize(sent)
    tagged = pos_tag(tokens)
    # Find and highlight 'record'
    for word, tag in tagged:
        if word.lower() == 'record':
            print(f"Sentence: \"{sent}\"")
            print(f"  'record' tagged as: {tag} ({tag_meanings.get(tag, tag)})")
            print()

Sentence: "Please record the meeting for later review."
  'record' tagged as: VB (Verb (base))

Sentence: "She broke the world record last year."
  'record' tagged as: NN (Noun (singular))



---
## Part 2 — Named Entity Recognition with NLTK's `ne_chunk`

**NER** identifies real-world entities in text. NLTK's built-in `ne_chunk` function uses the **MaxEnt chunker model** trained on the ACE corpus.

The pipeline is: **raw text → tokenise → POS tag → NE chunk**.

Key entity types recognised by NLTK:
- `PERSON` — people's names
- `GPE` — geopolitical entities (countries, cities)
- `ORGANIZATION` — company/institution names
- `LOCATION` — non-GPE locations (mountains, rivers)

In [6]:
from nltk import ne_chunk

text = "vincent van gogh was born in Netherlands."

# The full pipeline
tokens = word_tokenize(text)
tagged = pos_tag(tokens)
entities = ne_chunk(tagged)

print("Raw ne_chunk tree output:")
print(entities)

Raw ne_chunk tree output:
(S
  vincent/NN
  van/NN
  gogh/NN
  was/VBD
  born/VBN
  in/IN
  (GPE Netherlands/NNP)
  ./.)


### 2.1 Extracting entities cleanly

The `ne_chunk` output is a tree. Named entities are `Tree` nodes with a label; plain words are `(word, tag)` tuples. Let's extract just the entities.

In [7]:
import nltk

def extract_entities(text):
    """Extract named entities from a text string."""
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    chunked = ne_chunk(tagged)

    entities = []
    for subtree in chunked:
        # Named entities are Tree nodes, not plain tuples
        if isinstance(subtree, nltk.Tree):
            entity_name = " ".join(word for word, tag in subtree.leaves())
            entity_type = subtree.label()
            entities.append((entity_name, entity_type))
    return entities


# Test on a richer sentence
test_sentences = [
    "Microsoft is a big company in Seattle.",
    "Barack Obama was born in Hawaii and attended Harvard University.",
    "The Amazon river flows through Brazil and Peru."
]

for sent in test_sentences:
    print(f"Text: {sent}")
    ents = extract_entities(sent)
    if ents:
        for name, etype in ents:
            print(f"  [{etype}] {name}")
    else:
        print("  (no entities found)")
    print()

Text: Microsoft is a big company in Seattle.
  [GPE] Microsoft
  [GPE] Seattle

Text: Barack Obama was born in Hawaii and attended Harvard University.
  [PERSON] Barack
  [PERSON] Obama
  [GPE] Hawaii
  [ORGANIZATION] Harvard University

Text: The Amazon river flows through Brazil and Peru.
  [ORGANIZATION] Amazon
  [PERSON] Brazil
  [PERSON] Peru



---
## Part 3 — Training a Custom MaxEnt NER Classifier

NLTK's built-in `ne_chunk` is a general-purpose model. For domain-specific NER (or when we want fine-grained control) we can train our own **Maximum Entropy classifier**.

### How MaxEnt works

The model assigns a probability to each label `y` given input features `x`:

$$P(y|x) = \frac{1}{Z(x)} \exp\left(\sum_i \lambda_i f_i(x, y)\right)$$

- **$f_i(x, y)$** — feature functions (e.g., "is the word capitalised?")  
- **$\lambda_i$** — weights learned during training  
- **$Z(x)$** — normalisation constant so probabilities sum to 1

### Step 1 — Define a feature extraction function

Features capture useful signals for entity detection:

In [8]:
def extract_features(word, index, sentence):
    """
    Extract a feature dictionary for a single word in its sentence context.

    Args:
        word     : the current word string
        index    : position of the word in the sentence list
        sentence : list of all word strings in the sentence

    Returns:
        dict of feature_name -> feature_value
    """
    features = {
        # The word itself (lowercased for generalisation)
        'word': word.lower(),

        # Does it start with an uppercase letter? (strong signal for proper nouns)
        'is_capitalized': word[0].isupper(),

        # Is the entire word uppercase? (e.g. abbreviations like USA)
        'is_all_caps': word.isupper(),

        # Does the word contain a digit? (useful for dates, quantities)
        'contains_digit': any(c.isdigit() for c in word),

        # Word suffix — last 2 and 3 characters (morphological clue)
        'suffix_2': word[-2:] if len(word) >= 2 else word,
        'suffix_3': word[-3:] if len(word) >= 3 else word,

        # Word prefix — first 2 characters
        'prefix_2': word[:2] if len(word) >= 2 else word,

        # Word length as a rough size signal
        'word_length': len(word),

        # Neighbouring words — context is crucial for NER
        'prev_word': sentence[index - 1].lower() if index > 0 else '<START>',
        'next_word': sentence[index + 1].lower() if index < len(sentence) - 1 else '<END>',

        # Is this the first word in the sentence?
        'is_first': index == 0,

        # Is this the last word?
        'is_last': index == len(sentence) - 1,
    }
    return features


# Demo: inspect features for 'Vincent' in context
demo_sentence = ['Vincent', 'Van', 'Gogh', 'was', 'born', 'in', 'the', 'Netherlands']
print("Features for 'Vincent':")
for k, v in extract_features('Vincent', 0, demo_sentence).items():
    print(f"  {k:20s}: {v}")

Features for 'Vincent':
  word                : vincent
  is_capitalized      : True
  is_all_caps         : False
  contains_digit      : False
  suffix_2            : nt
  suffix_3            : ent
  prefix_2            : Vi
  word_length         : 7
  prev_word           : <START>
  next_word           : van
  is_first            : True
  is_last             : False


### Step 2 — Build the training dataset

Each training example is a `(feature_dict, label)` pair. Labels follow the **BIO scheme**:

| Label | Meaning |
|-------|---------|
| `B-PERSON` | Beginning of a person entity |
| `I-PERSON` | Inside/continuation of a person entity |
| `B-GPE` | Beginning of a geopolitical entity |
| `I-GPE` | Inside a geopolitical entity |
| `B-ORG` | Beginning of an organisation |
| `O` | Outside — not an entity |

BIO tagging lets us correctly reconstruct multi-word entities like "Van Gogh" or "New York".

In [9]:
from nltk.tokenize import word_tokenize

# ---------------------------------------------------------------
# Labelled training sentences
# Format: (sentence_string, {word_index: BIO_label})
# Words not in the dict are labelled 'O' (outside)
# ---------------------------------------------------------------
labelled_corpus = [
    (
        "Vincent Van Gogh was born in the Netherlands.",
        {0: 'B-PERSON', 1: 'I-PERSON', 2: 'I-PERSON', 7: 'B-GPE'}
    ),
    (
        "Microsoft is a big company in Seattle.",
        {0: 'B-ORG', 6: 'B-GPE'}
    ),
    (
        "Barack Obama attended Harvard University in Cambridge.",
        {0: 'B-PERSON', 1: 'I-PERSON', 2: 'O', 3: 'B-ORG', 4: 'I-ORG', 6: 'B-GPE'}
    ),
    (
        "Apple was founded by Steve Jobs in Cupertino California.",
        {0: 'B-ORG', 4: 'B-PERSON', 5: 'I-PERSON', 7: 'B-GPE', 8: 'I-GPE'}
    ),
    (
        "Elon Musk is the CEO of Tesla and SpaceX.",
        {0: 'B-PERSON', 1: 'I-PERSON', 6: 'B-ORG', 8: 'B-ORG'}
    ),
    (
        "The Eiffel Tower is located in Paris France.",
        {1: 'B-GPE', 2: 'I-GPE', 6: 'B-GPE', 7: 'B-GPE'}
    ),
    (
        "Google opened a new office in London.",
        {0: 'B-ORG', 6: 'B-GPE'}
    ),
    (
        "Albert Einstein was a physicist born in Germany.",
        {0: 'B-PERSON', 1: 'I-PERSON', 7: 'B-GPE'}
    ),
]


def build_training_data(labelled_corpus):
    """Convert labelled corpus into (feature_dict, label) pairs for MaxEnt."""
    training_data = []
    for sentence_str, label_map in labelled_corpus:
        tokens = word_tokenize(sentence_str)
        for i, word in enumerate(tokens):
            # Assign the label; default to 'O' if not in label_map
            label = label_map.get(i, 'O')
            features = extract_features(word, i, tokens)
            training_data.append((features, label))
    return training_data


training_data = build_training_data(labelled_corpus)

# Count label distribution
from collections import Counter
label_counts = Counter(label for _, label in training_data)
print("Training samples:", len(training_data))
print("Label distribution:")
for label, count in sorted(label_counts.items()):
    print(f"  {label:12s}: {count}")

Training samples: 71
Label distribution:
  B-GPE       : 9
  B-ORG       : 6
  B-PERSON    : 5
  I-GPE       : 2
  I-ORG       : 1
  I-PERSON    : 6
  O           : 42


### Step 3 — Train the MaxEnt classifier

`nltk.MaxentClassifier.train()` uses **Generalised Iterative Scaling (GIS)** to optimise the feature weights. We pass `max_iter` to control training duration.

In [10]:
# Train the MaxEnt classifier
# max_iter controls the number of GIS optimisation iterations
classifier = nltk.MaxentClassifier.train(
    training_data,
    max_iter=20,
    trace=0          # set trace=1 to see per-iteration loss
)

print("Training complete!")
print("\nTop 10 most informative features:")
classifier.show_most_informative_features(10)

Training complete!

Top 10 most informative features:
  -2.449 is_first==False and label is 'B-PERSON'
  -1.976 is_capitalized==True and label is 'O'
   1.311 word=='university' and label is 'I-ORG'
   1.311 suffix_2=='ty' and label is 'I-ORG'
   1.311 suffix_3=='ity' and label is 'I-ORG'
   1.311 prefix_2=='Un' and label is 'I-ORG'
   1.311 prev_word=='harvard' and label is 'I-ORG'
   1.229 word=='california' and label is 'I-GPE'
   1.229 suffix_2=='ia' and label is 'I-GPE'
   1.229 suffix_3=='nia' and label is 'I-GPE'


### Step 4 — Test the classifier

In [11]:
def predict_entities(classifier, sentence_str):
    """Run the trained classifier on a new sentence and print predictions."""
    tokens = word_tokenize(sentence_str)
    test_features = [extract_features(w, i, tokens) for i, w in enumerate(tokens)]
    predicted_tags = classifier.classify_many(test_features)

    print(f"Sentence: {sentence_str}")
    print(f"{'Word':<18} {'Predicted Tag'}")
    print("-" * 35)
    for word, tag in zip(tokens, predicted_tags):
        marker = " ◀" if tag != 'O' else ""
        print(f"{word:<18} {tag}{marker}")
    print()


# Test sentences (not seen during training)
predict_entities(classifier, "Microsoft is a big company in Seattle.")
predict_entities(classifier, "Taylor Swift performed in New York last week.")
predict_entities(classifier, "Amazon was founded by Jeff Bezos in Bellevue Washington.")

Sentence: Microsoft is a big company in Seattle.
Word               Predicted Tag
-----------------------------------
Microsoft          B-ORG ◀
is                 O
a                  O
big                O
company            O
in                 O
Seattle            B-GPE ◀
.                  O

Sentence: Taylor Swift performed in New York last week.
Word               Predicted Tag
-----------------------------------
Taylor             B-PERSON ◀
Swift              B-ORG ◀
performed          O
in                 O
New                B-GPE ◀
York               O
last               O
week               O
.                  O

Sentence: Amazon was founded by Jeff Bezos in Bellevue Washington.
Word               Predicted Tag
-----------------------------------
Amazon             B-PERSON ◀
was                O
founded            O
by                 O
Jeff               O
Bezos              O
in                 O
Bellevue           B-GPE ◀
Washington         B-GPE ◀
.                  

---
## Exercise 1 — POS Tagging & NER on a Reddit-style Dataset

The lab asks us to apply POS tagging and NER to the Reddit dataset from Lab 1.  
We simulate this with realistic Reddit posts across different subreddits (topics).  
Then we extend the MaxEnt classifier with **additional feature functions**.

### 1A — Simulate a Reddit dataset

In [12]:
import pandas as pd

# Simulated Reddit posts across different subreddits
reddit_posts = [
    # r/technology
    {"subreddit": "technology",
     "text": "Apple just announced the new iPhone 16 in Cupertino California. Tim Cook presented alongside the Vision Pro team."},
    {"subreddit": "technology",
     "text": "Microsoft acquired Activision Blizzard for 68.7 billion dollars, making it the biggest gaming deal in history."},
    {"subreddit": "technology",
     "text": "Google DeepMind released Gemini Ultra, their most powerful AI model to date, trained on 1.56 trillion tokens."},

    # r/worldnews
    {"subreddit": "worldnews",
     "text": "The European Union reached a deal with China on Tuesday after talks in Brussels lasted over 12 hours."},
    {"subreddit": "worldnews",
     "text": "Ursula von der Leyen and Emmanuel Macron met in Paris to discuss the future of NATO funding arrangements."},
    {"subreddit": "worldnews",
     "text": "Japan and South Korea signed a new trade agreement in Tokyo, aiming to boost exports by 30 percent."},

    # r/science
    {"subreddit": "science",
     "text": "Researchers at MIT discovered a new protein that may help treat Alzheimer's disease in elderly patients."},
    {"subreddit": "science",
     "text": "NASA's Artemis mission is scheduled to land astronauts on the Moon by 2026 near the South Pole."},
    {"subreddit": "science",
     "text": "A study published in Nature found that regular exercise reduces depression risk by approximately 43 percent."},

    # r/sports
    {"subreddit": "sports",
     "text": "Lionel Messi scored twice as Inter Miami defeated Los Angeles FC 3-1 in the MLS cup final."},
    {"subreddit": "sports",
     "text": "Novak Djokovic won his 24th Grand Slam title at Wimbledon defeating Carlos Alcaraz in five sets."},
]

df = pd.DataFrame(reddit_posts)
print(f"Dataset: {len(df)} posts across {df['subreddit'].nunique()} subreddits")
df

Dataset: 11 posts across 4 subreddits


,subreddit,text
0,technology,Apple just announced the new iPhone 16 in Cupe...
1,technology,Microsoft acquired Activision Blizzard for 68....
2,technology,"Google DeepMind released Gemini Ultra, their m..."
3,worldnews,The European Union reached a deal with China o...
4,worldnews,Ursula von der Leyen and Emmanuel Macron met i...
5,worldnews,Japan and South Korea signed a new trade agree...
6,science,Researchers at MIT discovered a new protein th...
7,science,NASA's Artemis mission is scheduled to land as...
8,science,A study published in Nature found that regular...
9,sports,Lionel Messi scored twice as Inter Miami defea...


### 1B — Apply POS Tagging to Reddit posts

We tag every post and analyse which POS categories are most frequent per subreddit.

In [13]:
from collections import defaultdict

def pos_tag_text(text):
    """Return list of (word, tag) pairs for a text string."""
    tokens = word_tokenize(text)
    return pos_tag(tokens)


# Apply POS tagging to every post
df['pos_tags'] = df['text'].apply(pos_tag_text)

# Show results for the first post in each subreddit
for sub in df['subreddit'].unique():
    row = df[df['subreddit'] == sub].iloc[0]
    print(f"=== r/{sub} ===")
    print(f"Text: {row['text'][:80]}...")
    print("POS Tags:")
    for word, tag in row['pos_tags']:
        print(f"  {word}/{tag}", end="  ")
    print("\n")

=== r/technology ===
Text: Apple just announced the new iPhone 16 in Cupertino California. Tim Cook present...
POS Tags:
  Apple/NNP    just/RB    announced/VBD    the/DT    new/JJ    iPhone/NN    16/CD    in/IN    Cupertino/NNP    California/NNP    ./.    Tim/NNP    Cook/NNP    presented/VBD    alongside/IN    the/DT    Vision/NNP    Pro/NNP    team/NN    ./.  

=== r/worldnews ===
Text: The European Union reached a deal with China on Tuesday after talks in Brussels ...
POS Tags:
  The/DT    European/NNP    Union/NNP    reached/VBD    a/DT    deal/NN    with/IN    China/NNP    on/IN    Tuesday/NNP    after/IN    talks/NNS    in/IN    Brussels/NNP    lasted/VBD    over/IN    12/CD    hours/NNS    ./.  

=== r/science ===
Text: Researchers at MIT discovered a new protein that may help treat Alzheimer's dise...
POS Tags:
  Researchers/NNS    at/IN    MIT/NNP    discovered/VBD    a/DT    new/JJ    protein/NN    that/WDT    may/MD    help/VB    treat/VB    Alzheimer/NNP    's/POS    diseas

In [14]:
# ---- Analyse POS distribution per subreddit ----

subreddit_pos = defaultdict(Counter)

for _, row in df.iterrows():
    for word, tag in row['pos_tags']:
        # Group into broad categories
        if tag.startswith('NN'):
            subreddit_pos[row['subreddit']]['NOUN'] += 1
        elif tag.startswith('VB'):
            subreddit_pos[row['subreddit']]['VERB'] += 1
        elif tag.startswith('JJ'):
            subreddit_pos[row['subreddit']]['ADJECTIVE'] += 1
        elif tag.startswith('RB'):
            subreddit_pos[row['subreddit']]['ADVERB'] += 1
        elif tag == 'NNP' or tag == 'NNPS':
            subreddit_pos[row['subreddit']]['PROPER_NOUN'] += 1

print(f"{'Subreddit':<15} {'NOUN':>8} {'VERB':>8} {'ADJ':>8} {'ADV':>8} {'PROPER':>8}")
print("-" * 60)
for sub, counts in subreddit_pos.items():
    print(f"{sub:<15} {counts['NOUN']:>8} {counts['VERB']:>8} "
          f"{counts['ADJECTIVE']:>8} {counts['ADVERB']:>8} {counts['PROPER_NOUN']:>8}")

Subreddit           NOUN     VERB      ADJ      ADV   PROPER
------------------------------------------------------------
technology            24        6        3        2        0
worldnews             26        8        1        0        0
science               20        8        3        1        0
sports                18        4        1        1        0


### 1C — Apply NER to Reddit posts

We extract all named entities from each post and count what types appear most per subreddit.

In [15]:
def get_named_entities(text):
    """Return list of (entity_text, entity_type) from a raw text string."""
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    chunked = ne_chunk(tagged)
    entities = []
    for subtree in chunked:
        if isinstance(subtree, nltk.Tree):
            name = " ".join(w for w, t in subtree.leaves())
            entities.append((name, subtree.label()))
    return entities


df['entities'] = df['text'].apply(get_named_entities)

# Print all found entities per post
print("Named Entities found per post:\n")
for _, row in df.iterrows():
    print(f"[r/{row['subreddit']}] {row['text'][:60]}...")
    if row['entities']:
        for name, etype in row['entities']:
            print(f"   [{etype}] {name}")
    else:
        print("   (none detected)")
    print()

Named Entities found per post:

[r/technology] Apple just announced the new iPhone 16 in Cupertino Californ...
   [GPE] Apple
   [GPE] Cupertino California
   [PERSON] Tim Cook
   [ORGANIZATION] Vision Pro

[r/technology] Microsoft acquired Activision Blizzard for 68.7 billion doll...
   [PERSON] Microsoft
   [ORGANIZATION] Activision Blizzard

[r/technology] Google DeepMind released Gemini Ultra, their most powerful A...
   [PERSON] Google
   [ORGANIZATION] DeepMind
   [PERSON] Gemini Ultra

[r/worldnews] The European Union reached a deal with China on Tuesday afte...
   [ORGANIZATION] European Union
   [GPE] China
   [GPE] Brussels

[r/worldnews] Ursula von der Leyen and Emmanuel Macron met in Paris to dis...
   [PERSON] Ursula
   [PERSON] Leyen
   [PERSON] Emmanuel Macron
   [GPE] Paris
   [ORGANIZATION] NATO

[r/worldnews] Japan and South Korea signed a new trade agreement in Tokyo,...
   [GPE] Japan
   [GPE] South Korea
   [GPE] Tokyo

[r/science] Researchers at MIT discovered a n

### 1D — Extended MaxEnt Classifier with more features

The lab asks us to **define more feature functions**. We add:
- POS tag of the current word (from NLTK's tagger)
- POS tag of prev/next words
- Whether the word is a known English word (helps flag proper nouns)
- 2-character word prefix
- Whether the word appears after a title word like "Mr", "Dr", "President"

In [16]:
from nltk.corpus import words as nltk_words
nltk.download('words', quiet=True)

ENGLISH_WORDS = set(nltk_words.words())
TITLE_WORDS   = {'mr', 'mrs', 'ms', 'dr', 'prof', 'president', 'senator',
                 'minister', 'director', 'ceo', 'founder', 'chairman'}


def extract_features_v2(word, index, sentence, pos_tags):
    """
    Extended feature extractor — v2.
    Adds POS tag context and lexical knowledge features on top of v1.

    Args:
        word      : current word string
        index     : position in sentence
        sentence  : list of word strings
        pos_tags  : list of (word, tag) POS pairs for the sentence
    """
    current_pos = pos_tags[index][1]
    prev_pos    = pos_tags[index - 1][1] if index > 0 else '<START>'
    next_pos    = pos_tags[index + 1][1] if index < len(sentence) - 1 else '<END>'
    prev_word   = sentence[index - 1].lower() if index > 0 else '<START>'

    features = {
        # --- Basic word shape ---
        'word':              word.lower(),
        'is_capitalized':    word[0].isupper(),
        'is_all_caps':       word.isupper(),
        'contains_digit':    any(c.isdigit() for c in word),
        'word_length':       len(word),
        'suffix_2':          word[-2:] if len(word) >= 2 else word,
        'suffix_3':          word[-3:] if len(word) >= 3 else word,
        'prefix_2':          word[:2],

        # --- Context words ---
        'prev_word':         prev_word,
        'next_word':         sentence[index + 1].lower() if index < len(sentence) - 1 else '<END>',
        'is_first':          index == 0,
        'is_last':           index == len(sentence) - 1,

        # --- POS context (NEW) ---
        'pos_tag':           current_pos,         # e.g. NNP strongly signals entity
        'prev_pos':          prev_pos,
        'next_pos':          next_pos,

        # --- Lexical knowledge (NEW) ---
        # Unknown words that are capitalised are likely proper nouns / entities
        'is_known_english':  word.lower() in ENGLISH_WORDS,

        # Preceded by a title word ("Dr Smith", "President Biden")
        'after_title':       prev_word in TITLE_WORDS,

        # Compound feature: capitalised AND unknown English word (very likely entity)
        'cap_and_unknown':   word[0].isupper() and word.lower() not in ENGLISH_WORDS,
    }
    return features


def build_training_data_v2(labelled_corpus):
    """Same as before but using the extended feature extractor."""
    training_data = []
    for sentence_str, label_map in labelled_corpus:
        tokens   = word_tokenize(sentence_str)
        pos_tags = pos_tag(tokens)
        for i, word in enumerate(tokens):
            label    = label_map.get(i, 'O')
            features = extract_features_v2(word, i, tokens, pos_tags)
            training_data.append((features, label))
    return training_data


training_data_v2 = build_training_data_v2(labelled_corpus)
print(f"Training samples (v2): {len(training_data_v2)}")

classifier_v2 = nltk.MaxentClassifier.train(
    training_data_v2,
    max_iter=20,
    trace=0
)
print("Training complete (v2)!")
print("\nTop 15 most informative features (v2):")
classifier_v2.show_most_informative_features(15)

Training samples (v2): 71
Training complete (v2)!

Top 15 most informative features (v2):
  -1.726 is_first==False and label is 'B-PERSON'
  -1.411 cap_and_unknown==True and label is 'O'
  -1.371 prev_pos=='NNP' and label is 'B-GPE'
  -1.254 is_capitalized==True and label is 'O'
  -1.151 cap_and_unknown==False and label is 'B-ORG'
   1.135 word=='university' and label is 'I-ORG'
   1.135 suffix_2=='ty' and label is 'I-ORG'
   1.135 suffix_3=='ity' and label is 'I-ORG'
   1.135 prefix_2=='Un' and label is 'I-ORG'
   1.135 prev_word=='harvard' and label is 'I-ORG'
   1.062 word_length==10 and label is 'I-ORG'
   1.022 word=='california' and label is 'I-GPE'
   1.022 suffix_2=='ia' and label is 'I-GPE'
   1.022 suffix_3=='nia' and label is 'I-GPE'
   1.022 prev_word=='cupertino' and label is 'I-GPE'


In [17]:
def predict_entities_v2(classifier, sentence_str):
    """Predict NER tags using the v2 feature extractor."""
    tokens   = word_tokenize(sentence_str)
    pos_tags_result = pos_tag(tokens)
    test_features = [
        extract_features_v2(w, i, tokens, pos_tags_result)
        for i, w in enumerate(tokens)
    ]
    predicted = classifier.classify_many(test_features)

    print(f"Text: {sentence_str}")
    print(f"{'Word':<18} {'POS':<8} {'NER Tag'}")
    print("-" * 40)
    for (word, pos), tag in zip(pos_tags_result, predicted):
        marker = " ◀" if tag != 'O' else ""
        print(f"{word:<18} {pos:<8} {tag}{marker}")
    print()


# Test on Reddit posts from our dataset
for _, row in df.head(4).iterrows():
    predict_entities_v2(classifier_v2, row['text'])

Text: Apple just announced the new iPhone 16 in Cupertino California. Tim Cook presented alongside the Vision Pro team.
Word               POS      NER Tag
----------------------------------------
Apple              NNP      B-ORG ◀
just               RB       O
announced          VBD      O
the                DT       O
new                JJ       O
iPhone             NN       O
16                 CD       O
in                 IN       O
Cupertino          NNP      B-GPE ◀
California         NNP      I-GPE ◀
.                  .        O
Tim                NNP      B-GPE ◀
Cook               NNP      O
presented          VBD      O
alongside          IN       O
the                DT       O
Vision             NNP      O
Pro                NNP      O
team               NN       O
.                  .        O

Text: Microsoft acquired Activision Blizzard for 68.7 billion dollars, making it the biggest gaming deal in history.
Word               POS      NER Tag
-------------------------

---
## Exercise 2 — Domain-specific NER with spaCy

For Exercise 2 we:
1. Use **spaCy** as an alternative NLP library for POS tagging and NER
2. Demonstrate **medical NER** using sample clinical text

### 2A — spaCy for general POS tagging and NER

spaCy processes text with a pre-trained neural pipeline in a single call — no manual tokenise → tag → chunk steps needed.

In [18]:
import spacy

# Load the small English model (downloaded in the setup cell)
nlp = spacy.load("en_core_web_sm")

text = "Microsoft CEO Satya Nadella announced new AI features at the Build conference in Seattle."
doc  = nlp(text)

# --- POS Tagging ---
print("=== POS Tagging (spaCy) ===")
print(f"{'Token':<18} {'POS':<8} {'Fine-grained POS':<18} {'Dependency'}")
print("-" * 60)
for token in doc:
    print(f"{token.text:<18} {token.pos_:<8} {token.tag_:<18} {token.dep_}")

print()

# --- NER ---
print("=== Named Entities (spaCy) ===")
print(f"{'Entity':<25} {'Label':<15} {'Description'}")
print("-" * 60)
for ent in doc.ents:
    print(f"{ent.text:<25} {ent.label_:<15} {spacy.explain(ent.label_)}")

=== POS Tagging (spaCy) ===
Token              POS      Fine-grained POS   Dependency
------------------------------------------------------------
Microsoft          PROPN    NNP                compound
CEO                PROPN    NNP                compound
Satya              PROPN    NNP                compound
Nadella            PROPN    NNP                nsubj
announced          VERB     VBD                ROOT
new                ADJ      JJ                 amod
AI                 PROPN    NNP                compound
features           NOUN     NNS                dobj
at                 ADP      IN                 prep
the                DET      DT                 det
Build              PROPN    NNP                compound
conference         NOUN     NN                 pobj
in                 ADP      IN                 prep
Seattle            PROPN    NNP                pobj
.                  PUNCT    .                  punct

=== Named Entities (spaCy) ===
Entity              

In [19]:
# Apply spaCy NER across the full Reddit dataset and compare with NLTK

print("Comparing NLTK vs spaCy NER on Reddit posts:\n")
print("=" * 70)

for _, row in df.iterrows():
    text = row['text']
    print(f"[r/{row['subreddit']}] {text[:65]}...")

    # NLTK entities (already computed)
    nltk_ents = row['entities']
    print(f"  NLTK : {[(e, t) for e, t in nltk_ents] if nltk_ents else 'none'}")

    # spaCy entities
    doc = nlp(text)
    spacy_ents = [(ent.text, ent.label_) for ent in doc.ents]
    print(f"  spaCy: {spacy_ents if spacy_ents else 'none'}")
    print()

Comparing NLTK vs spaCy NER on Reddit posts:

[r/technology] Apple just announced the new iPhone 16 in Cupertino California. T...
  NLTK : [('Apple', 'GPE'), ('Cupertino California', 'GPE'), ('Tim Cook', 'PERSON'), ('Vision Pro', 'ORGANIZATION')]
  spaCy: [('Apple', 'ORG'), ('16', 'CARDINAL'), ('Cupertino California', 'GPE'), ('Tim Cook', 'PERSON')]

[r/technology] Microsoft acquired Activision Blizzard for 68.7 billion dollars, ...
  NLTK : [('Microsoft', 'PERSON'), ('Activision Blizzard', 'ORGANIZATION')]
  spaCy: [('Microsoft', 'ORG'), ('Activision Blizzard', 'PERSON'), ('68.7 billion dollars', 'MONEY')]

[r/technology] Google DeepMind released Gemini Ultra, their most powerful AI mod...
  NLTK : [('Google', 'PERSON'), ('DeepMind', 'ORGANIZATION'), ('Gemini Ultra', 'PERSON')]
  spaCy: [('Google DeepMind', 'ORG'), ('Gemini Ultra', 'PERSON'), ('AI', 'GPE'), ('1.56 trillion', 'CARDINAL')]

[r/worldnews] The European Union reached a deal with China on Tuesday after tal...
  NLTK : [('Eu

### 2B — Medical Domain NER with spaCy

The lab suggests training a **domain-specific NER** model. Here we demonstrate using spaCy's **rule-based EntityRuler** to add medical entity patterns (DISEASE, DRUG, SYMPTOM) on top of the existing model.

This approach is practical when you have domain knowledge but limited labelled training data — you define patterns directly.

In [20]:
import spacy
from spacy.pipeline import EntityRuler

# Build a new spaCy pipeline with medical entity rules on top
nlp_med = spacy.load("en_core_web_sm")

# Add an EntityRuler BEFORE the NER component so it takes priority
ruler = nlp_med.add_pipe("entity_ruler", before="ner")

# Define medical patterns
# Each pattern: {"label": ENTITY_TYPE, "pattern": text_or_list_of_tokens}
medical_patterns = [
    # --- DISEASE ---
    {"label": "DISEASE", "pattern": "Alzheimer's disease"},
    {"label": "DISEASE", "pattern": "Alzheimer's"},
    {"label": "DISEASE", "pattern": "diabetes"},
    {"label": "DISEASE", "pattern": "type 2 diabetes"},
    {"label": "DISEASE", "pattern": "hypertension"},
    {"label": "DISEASE", "pattern": "depression"},
    {"label": "DISEASE", "pattern": "cancer"},
    {"label": "DISEASE", "pattern": "lung cancer"},
    {"label": "DISEASE", "pattern": "breast cancer"},
    {"label": "DISEASE", "pattern": "COVID-19"},
    {"label": "DISEASE", "pattern": "Parkinson's disease"},
    {"label": "DISEASE", "pattern": "asthma"},

    # --- DRUG ---
    {"label": "DRUG", "pattern": "ibuprofen"},
    {"label": "DRUG", "pattern": "paracetamol"},
    {"label": "DRUG", "pattern": "aspirin"},
    {"label": "DRUG", "pattern": "metformin"},
    {"label": "DRUG", "pattern": "amoxicillin"},
    {"label": "DRUG", "pattern": "lisinopril"},
    {"label": "DRUG", "pattern": "atorvastatin"},
    {"label": "DRUG", "pattern": "omeprazole"},
    {"label": "DRUG", "pattern": "sertraline"},

    # --- SYMPTOM ---
    {"label": "SYMPTOM", "pattern": "fever"},
    {"label": "SYMPTOM", "pattern": "fatigue"},
    {"label": "SYMPTOM", "pattern": "chest pain"},
    {"label": "SYMPTOM", "pattern": "shortness of breath"},
    {"label": "SYMPTOM", "pattern": "headache"},
    {"label": "SYMPTOM", "pattern": "nausea"},
    {"label": "SYMPTOM", "pattern": "vomiting"},
    {"label": "SYMPTOM", "pattern": "cough"},
    {"label": "SYMPTOM", "pattern": "memory loss"},
    {"label": "SYMPTOM", "pattern": "tremors"},

    # --- BODY_PART ---
    {"label": "BODY_PART", "pattern": "liver"},
    {"label": "BODY_PART", "pattern": "kidney"},
    {"label": "BODY_PART", "pattern": "lungs"},
    {"label": "BODY_PART", "pattern": "brain"},
    {"label": "BODY_PART", "pattern": "heart"},
]

ruler.add_patterns(medical_patterns)
print(f"Medical EntityRuler added with {len(medical_patterns)} patterns.")

Medical EntityRuler added with 36 patterns.


In [21]:
# Sample clinical/medical texts
medical_texts = [
    "The patient presented with fever, headache, and shortness of breath. "
    "A diagnosis of COVID-19 was confirmed after PCR testing.",

    "She was prescribed metformin 500mg for type 2 diabetes and lisinopril for hypertension.",

    "MRI scans revealed early signs of Alzheimer's disease. "
    "The neurologist recommended sertraline to manage depression symptoms.",

    "The 68-year-old male reported tremors, fatigue and memory loss. "
    "Parkinson's disease was suspected following the neurological examination.",

    "Post-operative nausea and vomiting were treated with ibuprofen and omeprazole."
]

print("Medical NER Results\n" + "=" * 60)

for text in medical_texts:
    doc = nlp_med(text)
    print(f"\nText: {text[:80]}...")
    print(f"{'Entity':<30} {'Label'}")
    print("-" * 45)
    for ent in doc.ents:
        print(f"{ent.text:<30} {ent.label_}")

Medical NER Results

Text: The patient presented with fever, headache, and shortness of breath. A diagnosis...
Entity                         Label
---------------------------------------------
fever                          SYMPTOM
headache                       SYMPTOM
shortness of breath            SYMPTOM
COVID-19                       DISEASE
PCR                            ORG

Text: She was prescribed metformin 500mg for type 2 diabetes and lisinopril for hypert...
Entity                         Label
---------------------------------------------
metformin                      DRUG
500                            CARDINAL
type 2 diabetes                DISEASE
lisinopril                     DRUG
hypertension                   DISEASE

Text: MRI scans revealed early signs of Alzheimer's disease. The neurologist recommend...
Entity                         Label
---------------------------------------------
Alzheimer's disease            DISEASE
sertraline                     DRUG
de

### 2C — spaCy POS tagging on medical text

In [22]:
clinical_note = (
    "The elderly patient was administered amoxicillin to treat a severe bacterial infection "
    "affecting the lungs. Symptoms included cough, fever, and chest pain."
)

doc = nlp_med(clinical_note)

print("=== POS Tags ===")
print(f"{'Token':<18} {'POS':<10} {'Tag'}")
print("-" * 40)
for token in doc:
    print(f"{token.text:<18} {token.pos_:<10} {token.tag_}")

print("\n=== Named Entities ===")
print(f"{'Entity':<25} {'Label'}")
print("-" * 35)
for ent in doc.ents:
    print(f"{ent.text:<25} {ent.label_}")

=== POS Tags ===
Token              POS        Tag
----------------------------------------
The                DET        DT
elderly            ADJ        JJ
patient            NOUN       NN
was                AUX        VBD
administered       VERB       VBN
amoxicillin        NOUN       NN
to                 PART       TO
treat              VERB       VB
a                  DET        DT
severe             ADJ        JJ
bacterial          ADJ        JJ
infection          NOUN       NN
affecting          VERB       VBG
the                DET        DT
lungs              NOUN       NNS
.                  PUNCT      .
Symptoms           NOUN       NNS
included           VERB       VBD
cough              NOUN       NN
,                  PUNCT      ,
fever              NOUN       NN
,                  PUNCT      ,
and                CCONJ      CC
chest              NOUN       NN
pain               NOUN       NN
.                  PUNCT      .

=== Named Entities ===
Entity                  

---
## Summary

| Task | Tool | Key Function |
|------|------|--------------|
| POS Tagging | NLTK | `pos_tag(word_tokenize(text))` |
| Basic NER | NLTK | `ne_chunk(tagged)` |
| Custom NER | NLTK MaxEnt | `MaxentClassifier.train(data)` |
| POS + NER (neural) | spaCy | `nlp(text)` → `.pos_`, `.ents` |
| Medical NER (rule-based) | spaCy EntityRuler | `ruler.add_patterns(patterns)` |

**Key takeaways:**
- POS tags are the foundation — many NER features rely on them (NNP = proper noun is a strong entity signal)
- The MaxEnt classifier is only as good as your labelled data and features — more data always helps
- spaCy's neural model is significantly more accurate than NLTK's `ne_chunk` on general text
- For domain-specific NER (medical, legal), combining a base model with rule-based patterns is a practical and efficient strategy